# Gaussian Mixture Model with EM — Solution

**Difficulty**: 🟡 Medium

---

### Problem Statement

The Gaussian mixture model is the canonical latent-variable model, and **expectation-maximization
(EM)** is the canonical way to fit it. GMMs are used for soft clustering, anomaly detection, color
and background segmentation, Fisher-vector image encoding, and codebook initialization. The EM
lower bound is also the same ELBO that VAEs and diffusion models optimize. The model is

$$p(x) = \sum_{k=1}^{K} \pi_k\, \mathcal N(x \mid \mu_k, \Sigma_k), \qquad \pi_k \ge 0,\ \sum_k \pi_k = 1 .$$

EM alternates two closed-form steps that never decrease the likelihood:

- **E-step**: the responsibilities $r_{nk} = p(k \mid x_n) = \dfrac{\pi_k \mathcal N(x_n\mid\mu_k,\Sigma_k)}{\sum_j \pi_j \mathcal N(x_n\mid\mu_j,\Sigma_j)}$.
- **M-step**: with $N_k = \sum_n r_{nk}$,
  $$\pi_k = \frac{N_k}{N},\qquad \mu_k = \frac{1}{N_k}\sum_n r_{nk}\,x_n,\qquad \Sigma_k = \frac{1}{N_k}\sum_n r_{nk}\,(x_n-\mu_k)(x_n-\mu_k)^{\top} + \texttt{reg\_covar}\cdot I .$$

In more than a couple of dimensions the densities underflow to 0 in floating point, so the
E-step has to be done in **log space**.

---

### Requirements

1. **`GaussianMixture(n_components, max_iter=100, tol=1e-4, reg_covar=1e-6, seed=0)`** with full covariances.
2. **`fit(X)`**: `X` is `(N, D)`. Returns `self` and sets
   - `weights_` `(K,)`, `means_` `(K, D)`, `covariances_` `(K, D, D)`;
   - `log_likelihood_history_`: a list of floats. Entry $i$ is the **mean log-likelihood per sample**
     $\frac1N\sum_n \log p(x_n)$ of the parameters **after $i$ M-steps**, so entry 0 is the
     initialization.
3. **Loop and stopping**: run at most `max_iter` M-steps. Stop as soon as an M-step improves the
   mean log-likelihood by **less than `tol`**. The history therefore has between 2 and
   `max_iter + 1` entries, and `log_likelihood_history_[-1] == score(X)` for the fitted model.
4. **E-step in log space**: $\log r_{nk} = \log\pi_k + \log\mathcal N(x_n\mid\mu_k,\Sigma_k) - \operatorname{logsumexp}_j(\cdot)$,
   with `torch.logsumexp`. Responsibilities must stay finite even for points far from every component.
5. **M-step** exactly as above: divide by $N_k$ (not $N$, not $N_k - 1$), use the *new* means in
   the covariance, and add `reg_covar` to the diagonal.
6. **Initialization**: any reasonable scheme. The reference picks `K` distinct random rows of `X`
   as means using `torch.Generator().manual_seed(seed)`, gives every component the (biased)
   covariance of `X` plus `reg_covar · I`, and starts with equal weights. Don't consume the
   global RNG.
7. **`predict_proba(X)`** → `(M, K)` responsibilities. **`predict(X)`** → `(M,)` argmax.
   **`score(X)`** → `float`: the mean log-likelihood per sample.

---

### Constraints

- ✅ `torch.linalg` (`cholesky`, `solve_triangular`, `slogdet`, …), `torch.logsumexp`, broadcasting / `einsum`.
- ✅ Vectorize over samples and components. A Python loop over EM iterations is fine.
- ❌ Don't use `torch.distributions` in your implementation (the tests use it as the oracle).
- ❌ Don't compute densities with `exp` and normalize afterwards: $e^{-1000} = 0$ gives $0/0$.
- ❌ No scikit-learn.

---

<details>
  <summary>💡 Hint</summary>

  - $\log \mathcal N(x\mid\mu,\Sigma) = -\tfrac12\big[D\log 2\pi + \log\lvert\Sigma\rvert + (x-\mu)^{\top}\Sigma^{-1}(x-\mu)\big]$.
    Which pieces depend on $x$, and which only on the component?
  - With a Cholesky factor $\Sigma = LL^{\top}$: $\log\lvert\Sigma\rvert = 2\sum_i \log L_{ii}$, and the
    Mahalanobis term is $\lVert z\rVert^2$ where $Lz = x - \mu$, which is a triangular solve.
  - `X[None] - means[:, None]` has shape `(K, N, D)`. Batched `cholesky` / `solve_triangular`
    handle all components at once.
  - The weighted covariance of all components is a single `einsum`.
  - Before writing the loop, decide which parameters each history entry belongs to.

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
# Three 2D Gaussians with different shapes and unequal weights.
torch.manual_seed(0)
true_weights = torch.tensor([0.5, 0.3, 0.2])
true_means = torch.tensor([[0.0, 0.0], [5.0, 1.0], [1.0, 5.0]])
true_covs = torch.tensor([[[1.0, 0.6], [0.6, 1.0]],
                          [[0.5, -0.2], [-0.2, 0.8]],
                          [[0.3, 0.0], [0.0, 0.3]]])
counts = (true_weights * 600).long().tolist()                          # 300, 180, 120
X = torch.cat([m + torch.randn(n, 2) @ torch.linalg.cholesky(c).T      # x = mu + L eps
               for m, c, n in zip(true_means, true_covs, counts)])
X = X[torch.randperm(len(X))]
print("X", tuple(X.shape))

In [ ]:
class GaussianMixture:
    """Gaussian mixture with full covariances, fitted by expectation-maximization."""

    def __init__(self, n_components, max_iter=100, tol=1e-4, reg_covar=1e-6, seed=0):
        self.n_components = n_components
        self.max_iter = max_iter
        self.tol = tol
        self.reg_covar = reg_covar
        self.seed = seed

    # ------------------------------------------------------------ internals
    def _log_joint(self, X):
        """log pi_k + log N(x_n | mu_k, Sigma_k) for every pair -> (N, K)."""
        N, D = X.shape
        L = torch.linalg.cholesky(self.covariances_)                    # (K, D, D), Sigma = L L^T
        diff = X.unsqueeze(0) - self.means_.unsqueeze(1)                # (K, N, D)
        z = torch.linalg.solve_triangular(L, diff.mT, upper=False)       # (K, D, N) = L^-1 (x - mu)
        mahalanobis = z.pow(2).sum(dim=1)                               # (K, N)
        log_det = 2 * torch.diagonal(L, dim1=-2, dim2=-1).log().sum(-1)  # (K,) log|Sigma_k|
        log_gauss = -0.5 * (D * math.log(2 * math.pi) + log_det[:, None] + mahalanobis)
        return (log_gauss + self.weights_.log()[:, None]).T              # (N, K)

    def _e_step(self, X):
        """Returns (mean log-likelihood per sample, responsibilities (N, K))."""
        log_joint = self._log_joint(X)
        log_px = torch.logsumexp(log_joint, dim=1, keepdim=True)        # (N, 1) log p(x_n)
        return float(log_px.mean()), torch.exp(log_joint - log_px)

    def _m_step(self, X, resp):
        N, D = X.shape
        nk = resp.sum(dim=0) + 10 * torch.finfo(X.dtype).eps            # (K,) soft counts
        self.weights_ = nk / nk.sum()
        self.means_ = resp.T @ X / nk[:, None]                          # (K, D)
        diff = X.unsqueeze(0) - self.means_.unsqueeze(1)                # (K, N, D)
        cov = torch.einsum("nk,knd,kne->kde", resp, diff, diff) / nk[:, None, None]
        self.covariances_ = cov + self.reg_covar * torch.eye(D, dtype=X.dtype)

    # ------------------------------------------------------------------ API
    def fit(self, X):
        """X: (N, D). Returns self."""
        N, D = X.shape
        K = self.n_components
        # Init: K distinct random points as means, the global covariance for every
        # component (broad, so each component can still claim its own cluster), equal weights.
        g = torch.Generator().manual_seed(self.seed)
        self.means_ = X[torch.randperm(N, generator=g)[:K]].clone()
        Xc = X - X.mean(dim=0)
        cov = Xc.T @ Xc / N + self.reg_covar * torch.eye(D, dtype=X.dtype)
        self.covariances_ = cov.expand(K, D, D).clone()
        self.weights_ = torch.full((K,), 1.0 / K, dtype=X.dtype)

        ll, resp = self._e_step(X)
        self.log_likelihood_history_ = [ll]             # entry i: after i M-steps
        for _ in range(self.max_iter):
            self._m_step(X, resp)
            new_ll, resp = self._e_step(X)
            self.log_likelihood_history_.append(new_ll)
            if new_ll - ll < self.tol:                  # EM never decreases the likelihood
                break
            ll = new_ll
        return self

    def predict_proba(self, X):
        """(M, D) -> (M, K) posterior p(component k | x)."""
        return self._e_step(X)[1]

    def predict(self, X):
        """(M, D) -> (M,) most likely component."""
        return self._log_joint(X).argmax(dim=1)

    def score(self, X):
        """Mean log-likelihood per sample, (1/M) sum_n log p(x_n)."""
        return self._e_step(X)[0]

In [ ]:
# Test 1: EM never decreases the likelihood; responsibilities are distributions, even far away
gmm = GaussianMixture(n_components=3, seed=0).fit(X)
h = gmm.log_likelihood_history_
print(f"{len(h) - 1} M-steps: mean log-likelihood {h[0]:.4f} -> {h[-1]:.4f}")
assert all(b >= a - 1e-6 for a, b in zip(h[:-1], h[1:])), "the log-likelihood decreased"
assert abs(h[-1] - gmm.score(X)) < 1e-4, "history[-1] must be the score of the final parameters"

R = gmm.predict_proba(X)
assert R.shape == (len(X), 3) and torch.allclose(R.sum(1), torch.ones(len(X)), atol=1e-5)
far = torch.tensor([[100.0, 100.0], [-500.0, 300.0]])
assert torch.isfinite(gmm.predict_proba(far)).all(), "NaN far from the data: do the E-step in log space"
assert torch.equal(gmm.predict(X), R.argmax(1))
print("Test 1 passed")

In [ ]:
# Test 2: score() against torch.distributions at the fitted parameters
gmm = GaussianMixture(n_components=3, seed=0).fit(X)
mvn = torch.distributions.MultivariateNormal(gmm.means_, covariance_matrix=gmm.covariances_)
log_joint = mvn.log_prob(X[:, None, :]) + gmm.weights_.log()            # (N, K)
reference = torch.logsumexp(log_joint, dim=1).mean().item()
print(f"score {gmm.score(X):.5f}  reference {reference:.5f}")
assert abs(gmm.score(X) - reference) < 1e-4
print("Test 2 passed")

In [ ]:
# Test 3: recover the generating parameters (best of 5 restarts, like sklearn's n_init)
best = max((GaussianMixture(3, seed=s).fit(X) for s in range(5)), key=lambda g: g.score(X))
match = torch.cdist(true_means, best.means_).argmin(dim=1)             # fitted component per true one
assert len(set(match.tolist())) == 3, "two true clusters were matched to the same component"
print("weights:", [round(v, 3) for v in best.weights_[match].tolist()], " true:", [round(v, 3) for v in true_weights.tolist()])
print("means  :", [[round(v, 2) for v in m] for m in best.means_[match].tolist()])
assert torch.allclose(best.weights_[match], true_weights, atol=0.03)
assert torch.allclose(best.means_[match], true_means, atol=0.2)
assert torch.allclose(best.covariances_[match], true_covs, atol=0.3)   # sampling noise, N = 600

# K = 1 is plain maximum likelihood: the sample mean and the biased sample covariance (+ reg)
one = GaussianMixture(n_components=1, reg_covar=1e-3).fit(X)
Xc = X - X.mean(0)
assert torch.allclose(one.means_[0], X.mean(0), atol=1e-4)
assert torch.allclose(one.covariances_[0], Xc.T @ Xc / len(X) + 1e-3 * torch.eye(2), atol=1e-4)
print("Test 3 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("gmm-em", GaussianMixture=GaussianMixture)

## Why it works

**EM maximizes a lower bound.** For any distributions $q_n(k)$, Jensen's inequality gives
$$\ell(\theta) = \sum_n \log \sum_k \pi_k \mathcal N(x_n\mid\theta_k) \;\ge\; \sum_n\sum_k q_{nk}\big[\log \pi_k\mathcal N(x_n\mid\theta_k) - \log q_{nk}\big] =: \mathcal L(q,\theta),$$
with equality when $q_{nk}$ is the posterior $r_{nk}$. The **E-step** sets $q = r(\theta_\text{old})$,
which makes the bound tight: $\mathcal L = \ell(\theta_\text{old})$. The **M-step** maximizes
$\mathcal L(q, \cdot)$, which separates into a weighted Gaussian MLE for each component plus a
Lagrange multiplier for $\sum_k \pi_k = 1$. That's where the closed forms and $\pi_k = N_k/N$
come from. Hence
$\ell(\theta_\text{new}) \ge \mathcal L(q,\theta_\text{new}) \ge \mathcal L(q,\theta_\text{old}) = \ell(\theta_\text{old})$:
the likelihood can never go down. That's the first test and the grader's monotonicity check. Adding
`reg_covar` makes the M-step a tiny perturbation of the exact maximizer, which is negligible here.

**Log space.** For a point 40 standard deviations from every component, each density is about
$e^{-800}$. That is 0 in float32 *and* float64, so normalizing gives $0/0$. With logs,
$\log r_{nk} = a_{nk} - \operatorname{logsumexp}_j a_{nj}$. `logsumexp` subtracts the row maximum
before exponentiating, so the largest term is $e^0 = 1$ and nothing underflows to an all-zero row.
The same `logsumexp` is the per-sample log-likelihood, which gives the score for free.

**Cholesky instead of `inv` + `det`.** $\Sigma = LL^{\top}$ costs $D^3/3$. The Mahalanobis term is
$\lVert L^{-1}(x-\mu)\rVert^2$, a triangular solve that is stabler than forming $\Sigma^{-1}$, and
$\log\lvert\Sigma\rvert = 2\sum_i\log L_{ii}$ can't overflow the way $\det$ does in high dimension.
All $K$ components go through one batched `cholesky` / `solve_triangular`, and the M-step
covariances are one `einsum` over the `(K, N, D)` difference tensor.

**History convention.** The reference evaluates the likelihood *after* each M-step (that E-step
is needed for the next M-step anyway). So `history[i]` belongs to the parameters after $i$ M-steps,
and the last entry is exactly `score(X)` of the returned model. Logging before the M-step, as
scikit-learn does internally, means the final parameters were never scored.

## Complexity

Per EM iteration: $O(NKD^2)$ for the Mahalanobis terms and the covariance `einsum`, plus $O(KD^3)$ for
the Cholesky factorizations. Memory is $O(NKD)$ for the difference tensor (process $N$ in chunks
if it's large). Convergence is linear: fast for well-separated clusters, slow when components overlap.

## Common bugs

- **Normalizing densities instead of log-densities**: NaN responsibilities for outliers or in high $D$.
- **Covariance divided by $N$ instead of $N_k$** (or by $N_k - 1$): no longer the M-step, and the
  likelihood can decrease.
- **Using the old means in the covariance update.** That covariance is too big by
  $(\mu_\text{new} - \mu_\text{old})(\mu_\text{new} - \mu_\text{old})^{\top}$. It is still a
  generalized EM step (the likelihood keeps rising) with the same fixed points, but it isn't the
  M-step's joint maximizer. The grader catches it with one M-step at $K = 1$.
- **Forgetting `reg_covar`**: Cholesky fails as soon as a component collapses onto a few
  points or the data lies in a subspace.
- **`score` returning the total instead of the mean, or leaving out $\log\pi_k$.**
- **History off by one**: logging before the M-step, so the returned parameters were never scored
  and the stopping test compares the wrong pair.
- **Hard assignments** (argmax responsibilities) in the M-step. That's classification EM / k-means,
  which doesn't monotonically increase the mixture likelihood.

## Interview follow-ups

1. **Does EM find the global maximum?** No. It converges monotonically to a stationary point
   (a local maximum or saddle) that depends on initialization. In practice: k-means++-style
   initialization, several restarts (`n_init`), and keep the highest likelihood. Convergence is
   linear, with a rate set by the fraction of missing information, so heavily overlapping components are slow.
2. **What happens with singular covariances?** The likelihood is unbounded. A component centered
   on a single point with $\Sigma_k \to 0$ sends $\ell \to \infty$. Remedies: `reg_covar`
   (diagonal loading), a MAP M-step with an inverse-Wishart prior, re-initializing collapsing
   components, or constraining covariances (tied / diagonal / spherical).
3. **How is k-means the hard-EM limit of a GMM?** Fix $\Sigma_k = \sigma^2 I$ and equal weights.
   Then $r_{nk} = \operatorname{softmax}_k(-\lVert x_n - \mu_k\rVert^2 / 2\sigma^2)$, which becomes a
   one-hot assignment to the nearest mean as $\sigma \to 0$, and the M-step mean becomes the
   cluster centroid: exactly Lloyd's algorithm.
4. **How do you choose $K$?** BIC/AIC with the parameter count $(K-1) + KD + KD(D+1)/2$, held-out
   log-likelihood, or a Bayesian GMM with a Dirichlet(-process) prior that empties unneeded components.
5. **Scaling to large $N$ or $D$?** Diagonal covariances ($O(NKD)$ per iteration), mini-batch / online
   EM that keeps running averages of the sufficient statistics, or SGD on the negative
   log-likelihood with a Cholesky parametrization (lets the GMM sit inside a neural network).
6. **How does this connect to VAEs?** It's the same bound. EM's E-step computes the *exact*
   posterior $q = p(k\mid x)$. When the posterior is intractable, a VAE *amortizes* it with an encoder
   $q_\phi(z\mid x)$ and maximizes the ELBO over $\phi$ and the decoder jointly by gradient descent.